# Risk QA Agent — Agent Node Testing

Tests the `agent_node` (entry point with self-retrieval via `create_agent`).

## Troubleshooting

- **Qdrant not running?**
  ```bash
  docker run -d --name nexlify-qdrant -p 6333:6333 -p 6334:6334 \
    -v $(pwd)/data/qdrant:/qdrant/storage qdrant/qdrant
  ```

- **Run this notebook:**
  ```bash
  uv run jupyter notebook notebooks/agent_node_testing.ipynb
  ```

In [ ]:
# === PHASE 0: Setup ===
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/lourvens/project/agentic-project/NexlifyCorp").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root: {PROJECT_ROOT}")

In [ ]:
# === PHASE 0: Environment ===
import os

from dotenv import load_dotenv
load_dotenv()

required = ["ANTHROPIC_API_KEY"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}")

print("✅ Environment loaded")
print(f"   ANTHROPIC_API_KEY: {os.getenv('ANTHROPIC_API_KEY')[:8]}...")

In [ ]:
# === PHASE 0: Config ===
from src.core.config import get_config

cfg = get_config()
print(f"✅ Config")
print(f"   Model: {cfg.anthropic_model}")
print(f"   Fast: {cfg.anthropic_fast_model}")

---

## Debug: Inspect create_agent output structure

First, let's see exactly what `create_agent` returns so we can debug the messages issue.

In [ ]:
# === DEBUG: Inspect create_agent output ===
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage
from src.core.llm import get_llm
from src.agents.tools import create_public_retriever_tool, create_private_retriever_tool
from src.agents.risk_qa_agent.prompts import AGENT_SYSTEM_PROMPT

AGENT_TOOLS = [
    create_public_retriever_tool(),
    create_private_retriever_tool(),
]

main_llm = get_llm()
agent = create_agent(
    model=main_llm,
    tools=AGENT_TOOLS,
    system_prompt=AGENT_SYSTEM_PROMPT,
)

# Simple conversational query — should NOT need tools
test_messages = [HumanMessage(content="who are you")]

result = agent.invoke({"messages": test_messages})

print("=== create_agent result keys ===")
print(list(result.keys()))
print()
print("=== messages in result ===")
msgs = result.get("messages", [])
print(f"Count: {len(msgs)}")
for i, m in enumerate(msgs):
    print(f"  [{i}] type={type(m).__name__}, hasattr content={hasattr(m, 'content')}")
    if hasattr(m, 'content'):
        content = m.content
        if isinstance(content, list):
            text = "".join(b.text for b in content if hasattr(b, 'text'))
        else:
            text = str(content)
        print(f"      content (first 100): {text[:100]}")
    if hasattr(m, 'type'):
        print(f"      type attribute: {m.type}")

---

## Test 1: Conversational query (should go DIRECT)

In [ ]:
# === TEST 1: Conversational query ===
from src.agents.risk_qa_agent.agent_node import agent_node
from langchain_core.messages import HumanMessage

state1 = {"messages": [HumanMessage(content="who are you")]}
result1 = agent_node(state1)

print("=== agent_node result ===")
print(f"needs_retrieval: {result1.get('needs_retrieval')}")
print(f"messages count: {len(result1.get('messages', []))}")
for i, m in enumerate(result1.get('messages', [])):
    print(f"  [{i}] {type(m).__name__}")
    if hasattr(m, 'content'):
        print(f"      content: {str(m.content)[:200]}")

---

## Test 2: NexlifyCorp query (should use retrieve_private_documents tool)

In [ ]:
# === TEST 2: NexlifyCorp internal query ===
state2 = {"messages": [HumanMessage(content="What is NexlifyCorp?")]}
result2 = agent_node(state2)

print("=== agent_node result ===")
print(f"needs_retrieval: {result2.get('needs_retrieval')}")
print(f"messages count: {len(result2.get('messages', []))}")
for i, m in enumerate(result2.get('messages', [])):
    print(f"  [{i}] {type(m).__name__}")
    if hasattr(m, 'content'):
        print(f"      content: {str(m.content)[:300]}")

---

## Test 3: SEC filings query (should use retrieve_public_documents tool)

In [ ]:
# === TEST 3: Public SEC query ===
state3 = {"messages": [HumanMessage(content="What risk factors does NVIDIA disclose in their 10-K?")]}
result3 = agent_node(state3)

print("=== agent_node result ===")
print(f"needs_retrieval: {result3.get('needs_retrieval')}")
print(f"messages count: {len(result3.get('messages', []))}")
for i, m in enumerate(result3.get('messages', [])):
    print(f"  [{i}] {type(m).__name__}")
    if hasattr(m, 'content'):
        print(f"      content: {str(m.content)[:300]}")

---

## Test 4: Complex query needing DELEGATE

In [ ]:
# === TEST 4: Complex comparison query (should DELEGATE) ===
state4 = {"messages": [HumanMessage(content="How does our Taiwan supply chain risk compare to what NVIDIA discloses in their SEC filings?")]}
result4 = agent_node(state4)

print("=== agent_node result ===")
print(f"needs_retrieval: {result4.get('needs_retrieval')}")
print(f"messages count: {len(result4.get('messages', []))}")
for i, m in enumerate(result4.get('messages', [])):
    print(f"  [{i}] {type(m).__name__}")
    if hasattr(m, 'content'):
        print(f"      content: {str(m.content)[:300]}")

---

## Full Graph: Agent Node as Entry Point

Run the full graph from START → agent_node with different query types.

In [ ]:
# === FULL GRAPH: Agent node entry point ===
from src.agents.risk_qa_agent.graph import get_risk_qa_graph

graph = get_risk_qa_graph()
print(f"✅ Graph nodes: {list(graph.nodes.keys())}")
print()

test_queries = [
    ("who are you", "conversational"),
    ("hello", "conversational"),
    ("What is NexlifyCorp?", "internal"),
    ("What risk factors does NVIDIA disclose?", "public"),
]

for query, label in test_queries:
    print(f"\n{'='*60}")
    print(f"Query [{label}]: {query}")
    print('='*60)
    try:
        result = graph.invoke({"messages": [HumanMessage(content=query)]})
        print(f"✅ needs_retrieval: {result.get('needs_retrieval')}")
        print(f"   route_key: {result.get('route_key', 'N/A')}")
        print(f"   messages count: {len(result.get('messages', []))}")
        if result.get('messages'):
            last = result['messages'][-1]
            content = last.content if hasattr(last, 'content') else str(last)
            if isinstance(content, list):
                content = "".join(b.text for b in content if hasattr(b, 'text'))
            print(f"   Answer (first 200): {content[:200]}")
        else:
            print("   ⚠️  NO MESSAGES IN RESULT")
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()